# 2026/8/21

## 人类反馈强化学习 (RLHF, Reinforcement Learning from Human Feedback)

## 为什么模型 SFT 后还需要用 RLHF

> **SFT 是“教模型标准答案怎么说”，RLHF 是“告诉模型什么样的回答更好”。**

<div align="center">

| 对比维度 | SFT（监督微调） | RLHF（基于人类反馈的强化学习） |
|---|---|---|
| **数据依赖** | 依赖大量高质量的人工标注数据 | 通过人类对模型输出进行评分、排序等反馈进行学习 |
| **泛化能力** | 标注数据**难以覆盖所有场景**，模型泛化能力有限 | 能够通过人类反馈学习更复杂、多样的偏好 |
| **创造性** | 容易模仿训练数据中的答案，缺乏主动优化能力 | 能够针对开放式任务学习多种合理答案，而不是局限于唯一标准答案 |
| **安全性** | 主要学习“应该输出什么”，对 **“不应该输出什么”** 的学习有限 | 可以通过负面示例和惩罚机制，让模型**学会拒绝**有害或不符合要求的输出 |
| **核心目标** | 学习“正确答案是什么” | 学习“什么样的答案更符合人类偏好” |
| **训练方式** | 给定输入和标准答案，直接计算监督学习损失 | 模型生成多个回答，由人类或奖励模型评价，再根据奖励优化模型 |

</div>
<br>

> 我认为 SFT 更像是给模型一个教材，模型从中学习给定的例题。如果例题很全面，能够讲考试内容覆盖到，模型就能获得更高分数，能力更强；
而 RLHF 像让一个学生直接从考试中学习，为了拿到更高的分数需要不断主动地理解考试内容、提升答题技巧，模型从被动学习变为主动学习

## KL 散度（Kullback-Leibler Divergence）

### 1. KL 散度是什么？

KL 散度用于**衡量两个概率分布之间的差异**，对于离散概率分布 $P$ 和 $Q$：

$$
D_{KL}(P|Q)= \sum_x P(x) \log \frac{P(x)}{Q(x)}
$$

其中：

* $P$：参考分布（Reference Distribution）
* $Q$：待比较的分布（Policy Distribution）
* $D_{KL}(P|Q)$：衡量 $Q$ 与 $P$ 的差异程度

> **如果真实情况是 $P$，但我们使用 $Q$ 来描述它，那么会产生多大的“信息损失”。**

KL 散度越小：

$$
P \approx Q
$$

---

### 2. KL 散度的计算

假设原始模型的概率分布为 $P$，经过强化学习之后，模型变成分布 $Q$：

<div align="center">

| Token |   P |    Q |
| ----- | --: | ---: |
| 好的    | 0.4 |  0.6 |
| 不错    | 0.3 |  0.2 |
| 很棒    | 0.2 | 0.15 |
| 差劲    | 0.1 | 0.05 |

</div>

根据计算公式逐项计算：

$$
0.4 \log \frac{0.4}{0.6}
\approx -0.162
$$

$$
0.3 \log \frac{0.3}{0.2}
\approx 0.122
$$

$$
0.2 \log \frac{0.2}{0.15}
\approx 0.058
$$

$$
0.1 \log \frac{0.1}{0.05}
\approx 0.069
$$

因此：

$$
D_{KL}(P|Q) \approx -0.162+0.122+0.058+0.069 \approx 0.087
$$

所以两个分布之间存在一定差异。

---

### 3. KL 散度数学性质

- 非负性

  $$
  D_{KL}(P|Q) \geq 0
  $$
  
  并且：
  
  $$
  D_{KL}(P|Q)=0 \iff P=Q
  $$
  
  这个结论可以通过 **詹森不等式（Jensen's Inequality）** 证明。

- 非对称性

  $$
  D_{KL}(P|Q) \neq D_{KL}(Q|P)
  $$

  因此 KL 散度不是严格意义上的“距离”。

---

### 4. KL 在 RLHF 中的作用

在 RLHF / PPO 等强化学习过程中，模型可能为了获得更高 reward 而发生**过度优化**。如果偏离过大，就可能出现：

* Reward hacking
* 输出质量下降
* 语言能力退化
* 模型行为不稳定
* 训练过度优化

因此可以加入 KL 惩罚，**使RL模型对于同一个输入产生的 token 概率分布不要与原模型的 token 概率分布太远**：

$$
L_{\text{total}}=L_{\text{RL}}+ \beta D_{KL}(P|Q)
$$

其中：

* Reward：希望模型获得更高奖励
* KL：限制模型不要偏离原始模型太远
* $\beta$：控制 KL 约束强度

---



## 奖励模型 (Reward Model)

> **奖励模型的任务不是生成回答，而是把人类对回答的偏好转化成一个标量分数。**

### 1. 奖励模型的作用

对于一个用户问题 $x$，LLM 会生成回答 $y$。将完整的“问题 + 回答”作为奖励模型的输入：

$$
(x,y) \longrightarrow R_\phi(x,y)
$$

奖励模型输出一个分数 $R_\phi(x,y)$，用来表示该回答在有用性、准确性、安全性以及是否符合人类偏好等方面的综合质量。好回答应得到更高分，差回答应得到更低分。

它的核心作用包括：

- 将难以直接写成规则的人类偏好量化为奖励信号；
- 在 RLHF 中替代人类持续打分，对模型生成的回答进行自动评价；
- 为后续 PPO 等强化学习算法提供优化方向。

> 奖励分数通常没有固定的绝对含义，重点是同一个问题下回答之间的相对高低。

---

### 2. Reward Model 与普通 LLM 的结构区别

两者可以使用相同的 Transformer 主体，主要区别在最后的输出头：

<div align="center">

| 对比项 | 普通 LLM（生成模型） | Reward Model（评分模型） |
|---|---|---|
| 输出头 | LM Head：`hidden_size × vocab_size` | Score Head：`hidden_size × 1` |
| 输出内容 | 下一个 token 在整个词表上的概率分布 | 当前“问题 + 回答”的单个奖励分数 |
| 训练目标 | 学习下一个 token | 学习人类对回答的偏好顺序 |
| 主要用途 | 文本生成、对话、问答 | 回答评价，为强化学习提供 reward |

</div>

因此代码中使用：

```python
AutoModelForSequenceClassification.from_pretrained(
    model_path,
    num_labels=1,
)
```

`num_labels=1` 表示模型对每条序列输出一个标量分数，由于训练中**损失函数大小取决于接受和拒绝的分数差异**，所以这里的 score 并不是普通意义上输出两个类别概率的二分类。

---

### 3. 为什么使用排序数据，而不是让人类直接打分

如果让不同标注员直接给回答打 1～10 分，容易出现以下问题：

- **主观性强**：不同人对同一个回答可能给出差别很大的分数；
- **评分标准不一致**：一个人的 7 分可能只相当于另一个人的 5 分；
- **训练噪声较大**：模型难以学习统一、稳定的绝对评分规则。

相比之下，让标注员判断“回答 A 和回答 B **哪个更好**”更加容易，也更容易达成共识。因此奖励模型通常使用成对偏好数据：

```python
{
    "prompt": "什么是数据库？",
    "chosen": "数据库是一个有组织的数据集合……",
    "rejected": "数据库用于存储数据。"
}
```

其中 `chosen` 是人类认为更好的回答，`rejected` 是相对较差的回答。奖励模型只需要学习：

$$
R_\phi(x,y_{chosen}) > R_\phi(x,y_{rejected})
$$

> 也就是“不怕不识货，就怕货比货”：比较相对优劣，通常比判断一个回答绝对值多少分更稳定。

---

### 4. 奖励模型的训练过程

对于每一组偏好数据，`RewardTrainer` 会完成以下计算：

1. 将 `prompt + chosen` 输入同一个奖励模型，得到 $r_{chosen}$；
2. 将 `prompt + rejected` 输入同一个奖励模型，得到 $r_{rejected}$；
3. 比较两个分数，并使用成对排序损失更新模型；
4. 让 $r_{chosen}$ 逐渐高于 $r_{rejected}$。

常用损失函数为：

$$
\mathcal{L}_{RM}
= -\log\sigma(r_{chosen}-r_{rejected})
$$

其中 $\sigma$ 是 Sigmoid 函数。这个损失关注的是两个回答的**分数差**：

- 当 $r_{chosen}=r_{rejected}$ 时，$\mathcal{L}= -\log(0.5)\approx0.693$，说明模型还不能区分优劣；
- 当 $r_{chosen}>r_{rejected}$ 且差距增大时，损失逐渐接近 0；
- 当 $r_{chosen}<r_{rejected}$ 时，说明排序错误，损失会变大。

所以奖励模型并不是学习“chosen=1、rejected=0”的普通分类，而是学习**同一问题下，优选回答的得分高于劣选回答**。

---

### 奖励模型训练（使用 LoRA ）

数据集链接：https://huggingface.co/datasets/BAAI/Infinity-Preference

任务类型：选择数据集中 task_category 中 logical_reasoning 这个逻辑推理任务

数据划分：该种任务有 4164 条数据，对其进行训练集 / 验证集 / 测试集 = 8 : 1 : 1 划分

模型配置

In [1]:
# 导入PyTorch深度学习框架
import torch
# 导入Hugging Face数据集处理库
from datasets import Dataset
# 导入JSON数据处理库
import json

# 导入PEFT（Parameter-Efficient Fine-Tuning）相关组件，用于高效微调
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
# 导入Transformers库相关组件，用于加载预训练模型和分词器
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForSequenceClassification
# 导入TRL（Transformer Reinforcement Learning）库，用于奖励模型训练
from trl import RewardTrainer, RewardConfig

# 定义预训练模型的路径（需要用户替换为实际路径）
model_path = "Qwen/Qwen2.5-0.5B"
# 从预训练模型路径加载分词器，禁用快速分词器以确保兼容性
tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False)
# 设置填充方向为右侧填充
tokenizer.padding_side = "right"
# 将结束标记设置为填充标记
tokenizer.pad_token = tokenizer.eos_token

# 配置4位量化参数，用于减少模型内存占用
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # 启用4位量化
    bnb_4bit_use_double_quant=True,  # 使用双重量化进一步压缩
    bnb_4bit_quant_type="nf4",  # 使用nf4量化类型
    bnb_4bit_compute_dtype=torch.float16  # 计算时使用float16精度
)

# 从预训练模型路径加载序列分类模型，配置为单标签分类
model = AutoModelForSequenceClassification.from_pretrained(model_path,
                                                           num_labels=1,  # 输出标签数为1（奖励分数）
                                                           quantization_config=bnb_config)  # 应用量化配置
# 设置模型的填充标记ID与分词器保持一致
model.config.pad_token_id = tokenizer.pad_token_id


W0822 09:59:26.685000 28592 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-0.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


LoRA 训练配置

In [2]:
# 配置LoRA（Low-Rank Adaptation）参数，用于高效微调
peft_config = LoraConfig(
    r=8,  # LoRA的秩，控制适配器的复杂度
    target_modules=["q_proj",  # 目标模块：查询投影层
                    "v_proj",  # 值投影层
                    "k_proj",  # 键投影层
                    "o_proj",  # 输出投影层
                    "gate_proj",  # 门控投影层
                    "down_proj",  # 下投影层
                    "up_proj"  # 上投影层
                    ],
    task_type=TaskType.SEQ_CLS,  # 任务类型：序列分类
    lora_alpha=16,  # LoRA的缩放参数
    lora_dropout=0.05  # LoRA的dropout率
)

# 为4位量化训练准备模型
model = prepare_model_for_kbit_training(model)
# 将LoRA适配器应用到模型上
model = get_peft_model(model, peft_config)
# 打印可训练参数的数量和比例
model.print_trainable_parameters()


trainable params: 4,400,000 || all params: 498,433,664 || trainable%: 0.8828


人类偏好数据集加载

In [3]:
from datasets import load_dataset

raw_dataset = load_dataset(
    "BAAI/Infinity-Preference",
    split="train",
)

print(raw_dataset)
print(raw_dataset[0])

Dataset({
    features: ['task_category', 'prompt', 'chosen', 'rejected', 'id'],
    num_rows: 59338
})
{'task_category': 'cryptography_knowledge', 'prompt': 'What is the procedure for utilizing Pretty Good Privacy (PGP) software on an Ubuntu operating system to create both RSA private and public encryption keys?', 'chosen': [{'content': 'What is the procedure for utilizing Pretty Good Privacy (PGP) software on an Ubuntu operating system to create both RSA private and public encryption keys?', 'role': 'user'}, {'content': 'Here\'s a comprehensive guide on generating RSA keys using PGP (GPG) on Ubuntu:\n\n**1. Installation**\n\n* **Install GPG:**\n\n   ```bash\n   sudo apt update\n   sudo apt install gnupg2\n   ```\n\n**2. Key Generation**\n\n* **Generate your keys:**\n\n   ```bash\n   gpg --full-generate-key\n   ```\n\n* **Follow the prompts:**\n\n   * **Key type:** Select "RSA and RSA" (for both encryption and signing).\n   * **Key size:**  Choose a strong size (at least 4096 bits is 

筛选出推理偏好数据

In [ ]:
from collections import Counter

print(raw_dataset.column_names)
print(Counter(raw_dataset["task_category"]).most_common())

reasoning_dataset = raw_dataset.filter(
    lambda x: x["task_category"] == "logical_reasoning"
)
print(reasoning_dataset)
print(reasoning_dataset[0])


['task_category', 'prompt', 'chosen', 'rejected', 'id']
[('logical_reasoning', 4164), ('problem_solved', 4059), ('answer_to_the_question', 1834), ('text_generation', 1341), ('programming_ability', 1287), ('mathematical_calculation', 1207), ('multicultural_understanding', 1012), ('writing_skills', 830), ('information_extraction', 711), ('knowledge_question_and_answer', 703), ('information_search', 641), ('search_knowledge', 639), ('creative_generation', 498), ('literary_creation', 475), ('historical_knowledge', 340), ('code_writing', 333), ('text_analysis', 297), ('command_understanding', 291), ('multiple_rounds_of_conversation', 277), ('translation', 274), ('health_knowledge', 269), ('humorous_understanding', 241), ('content_creation', 238), ('communication_skills', 223), ('data_analysis', 221), ('multitasking', 217), ('strategy_planning', 205), ('legal_knowledge', 202), ('knowledge_reasoning', 202), ('scriptwriting', 198), ('instruction_generation', 193), ('scene_construction', 193), 

Filter:   0%|          | 0/59338 [00:00<?, ? examples/s]

Dataset({
    features: ['task_category', 'prompt', 'chosen', 'rejected', 'id'],
    num_rows: 4164
})
{'task_category': 'logical_reasoning', 'prompt': 'Create a JavaScript function using a switch case that will receive a string of Roman numerals, transform them into normal numbers, and include error handling for invalid input.', 'chosen': [{'content': 'Create a JavaScript function using a switch case that will receive a string of Roman numerals, transform them into normal numbers, and include error handling for invalid input.', 'role': 'user'}, {'content': '```javascript\nfunction romanToInt(roman) {\n  // Create a mapping of Roman numerals to their integer values\n  const romanMap = {\n    \'I\': 1,\n    \'V\': 5,\n    \'X\': 10,\n    \'L\': 50,\n    \'C\': 100,\n    \'D\': 500,\n    \'M\': 1000,\n  };\n\n  let result = 0;\n\n  // Iterate through the Roman numeral string from right to left\n  for (let i = roman.length - 1; i >= 0; i--) {\n    const currentSymbol = roman[i];\n\n    //

数据划分 

训练集 / 验证集 / 测试集 = 8 : 1 : 1

In [5]:
# 固定随机种子，确保每次划分结果一致
all_dataset = reasoning_dataset.shuffle(seed=42)

# 先划出 20% 作为临时集合：后续再平分为验证集、测试集
split_1 = all_dataset.train_test_split(
    test_size=0.2,
    seed=42,
)

train_raw_dataset = split_1["train"]      # 80%
valid_test_dataset = split_1["test"]      # 20%

# 将剩下的 20% 再对半拆分
split_2 = valid_test_dataset.train_test_split(
    test_size=0.5,
    seed=42,
)

valid_raw_dataset = split_2["train"]      # 10%
test_raw_dataset = split_2["test"]        # 10%

print("训练集：", len(train_raw_dataset))
print("验证集：", len(valid_raw_dataset))
print("测试集：", len(test_raw_dataset))


训练集： 3331
验证集： 416
测试集： 417


数据集处理成对话模板形式

In [6]:
# 不需要像教程示例代码那样构造 chosen_id 和 chosen mask
# 定义数据预处理函数，将原始数据转换为模型训练所需的格式
def process_func(example):

    chosen_text = tokenizer.apply_chat_template(
        example["chosen"],
        tokenize=False,
        add_generation_prompt=False,
    )

    rejected_text = tokenizer.apply_chat_template(
        example["rejected"],
        tokenize=False,
        add_generation_prompt=False,
    )

    return {
        "chosen": chosen_text,
        "rejected": rejected_text,
    }


# 分别构造符合对话模板的格式，只有 chosen rejected 两个条目
train_dataset = train_raw_dataset.map(process_func, remove_columns=train_raw_dataset.column_names)
valid_dataset = valid_raw_dataset.map(process_func, remove_columns=train_raw_dataset.column_names)
test_dataset = test_raw_dataset.map(process_func, remove_columns=train_raw_dataset.column_names)

# 打印处理后的数据集信息
print(train_dataset)
print(valid_dataset)
print(test_dataset)

Map:   0%|          | 0/3331 [00:00<?, ? examples/s]

Map:   0%|          | 0/416 [00:00<?, ? examples/s]

Map:   0%|          | 0/417 [00:00<?, ? examples/s]

Dataset({
    features: ['chosen', 'rejected'],
    num_rows: 3331
})
Dataset({
    features: ['chosen', 'rejected'],
    num_rows: 416
})
Dataset({
    features: ['chosen', 'rejected'],
    num_rows: 417
})


In [7]:
# 配置奖励模型训练参数
config = RewardConfig(
    output_dir="../outputs/reasoning_reward_model",

    num_train_epochs=1,
    learning_rate=1e-5,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    logging_steps=100,

    # 每 100 step 在验证集上计算一次 eval_loss
    eval_strategy="steps",
    eval_steps=100,

    # 每 100 step 保存一次 checkpoint；必须与 eval_steps 对齐
    save_strategy="steps",
    save_steps=100,
    save_total_limit=4,

    # 训练结束后自动恢复验证集损失最低的 checkpoint
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none",
)

# 创建奖励模型训练器
trainer = RewardTrainer(
    model=model,  # 传入配置好的模型
    processing_class=tokenizer,  # 传入分词器
    args=config,  # 传入训练配置
    train_dataset=train_dataset,  # 传入训练数据集
    eval_dataset=valid_dataset
)


Adding EOS to train dataset:   0%|          | 0/3331 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3331 [00:00<?, ? examples/s]

Filtering train >1024 tokens:   0%|          | 0/3331 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/416 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/416 [00:00<?, ? examples/s]

Filtering eval >1024 tokens:   0%|          | 0/416 [00:00<?, ? examples/s]

### RewardTrainer 对偏好对的组批方式

> 值得注意的是，当前版本的 `RewardTrainer` 对输入数据的处理方式与部分旧版教程不同。

旧版教程通常要求我们提前对 `chosen` 和 `rejected` 分词，并在数据集中构造：

```python
input_ids_chosen
attention_mask_chosen
input_ids_rejected
attention_mask_rejected
```

当前版本只需要向 `RewardTrainer` 提供 `chosen`、`rejected` 两列文本，并通过 `processing_class=tokenizer` 让 Trainer 在内部完成分词和组批。

假设 `per_device_train_batch_size = B`，也就是一个 batch 中原本有 $B$ 组偏好对，数据整理器会把它们沿着 **batch 维度**排列为：

```text
input_ids[0:B]   -> B 条 chosen
input_ids[B:2B]  -> 与前面逐一对应的 B 条 rejected
```

因此送入模型的 `input_ids` 第一维大小是 $2B$，batch 中也只会看到统一的 `input_ids` 和 `attention_mask`，而不会看到 `input_ids_chosen`、`input_ids_rejected`。

> 这里的“拼接”是把 chosen 和 rejected **沿 batch 维排列**，不是把两个回答首尾连接成一条长文本。模型分别为它们打分，之后再将前后两半的分数配对，计算 $-\log\sigma(r_{chosen}-r_{rejected})$。

所以下面的 `pair_count` 表示当前 batch 中原始的偏好对数量：

```python
pair_count = input_ids.shape[0] // 2
```

In [8]:
batch = next(iter(trainer.get_train_dataloader()))

print(batch.keys())

input_ids = batch["input_ids"]
attention_mask = batch["attention_mask"].bool()

# RewardTrainer 将 chosen 与 rejected 沿 batch 维度拼接
pair_count = input_ids.shape[0] // 2 # 这里因为batch是1，只有一组接受/拒绝（input_ids.shape[0]=2），所以 pair_count=1
print(pair_count)
for i in range(pair_count):
    # 前半批：优选回答
    chosen_ids = input_ids[i][attention_mask[i]]

    # 后半批：与第 i 个 chosen 对应的劣选回答
    rejected_ids = input_ids[i + pair_count][attention_mask[i + pair_count]]

    print(f"\n========== 偏好对 {i} ==========")

    print("\n【chosen：优选回答】")
    print(tokenizer.decode(
        chosen_ids,
        skip_special_tokens=False,
    ))

    print("\n【rejected：劣选回答】")
    print(tokenizer.decode(
        rejected_ids,
        skip_special_tokens=False,
    ))

dict_keys(['input_ids', 'attention_mask'])
4

========== 偏好对 0 ==========

【chosen：优选回答】
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Could you explain the mathematical significance of both uppercase 'P' and lowercase 'p,' including at least one example for each?<|im_end|>
<|im_start|>assistant
Let's break down the mathematical significance of uppercase 'P' and lowercase 'p'.

**Uppercase P (P):**

* **Probability (of an Event):** In probability theory, uppercase 'P' often represents the probability of an event occurring. 

   * **Example:** The probability of rolling a six on a standard six-sided die is P(six) = 1/6. This means there's a 1 out of 6 chance of rolling a six.

* **Population:** In statistics, P sometimes refers to the entire population being studied.

   * **Example:** P might represent the total number of people living in a particular city.


**Lowercase p:**

* **Probability (of a Specific Event):**  Lowercase 'p' is often used when referr

In [9]:

# 开始训练奖励模型
trainer.train()
# 保存训练好的奖励模型到指定目录
trainer.save_model("../outputs/reward_model")


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss,Num Tokens,Min Reward,Mean Reward,Max Reward,Accuracy,Margin
100,0.727899,0.634934,391328.000000,-2.450277,-1.192745,0.304583,0.631076,0.312282
200,0.680849,0.601994,789093.000000,-2.062673,-0.453779,1.314317,0.657118,0.441744
300,0.657236,0.606391,1188294.000000,-1.841404,0.104793,2.059461,0.671875,0.507038
400,0.619441,0.565069,1589832.000000,-1.383684,0.764857,2.844482,0.694444,0.625719
500,0.669298,0.559925,1979788.000000,-0.650219,1.198775,2.966390,0.699653,0.535312
600,0.576695,0.551970,2363428.000000,-0.883377,1.164887,3.133138,0.702257,0.624712
700,0.594021,0.550374,2758566.000000,-0.912868,1.179215,3.178630,0.702257,0.635901
741,0.594021,0.549509,2917095.000000,-0.989120,1.099295,3.105713,0.707465,0.639894


使用测试集对训练结果评估

In [12]:
print(trainer.state.best_model_checkpoint)

../outputs/reasoning_reward_model\checkpoint-741


In [16]:
import torch
from tqdm.auto import tqdm

from peft import PeftModel
from transformers import AutoModelForSequenceClassification

base_model = AutoModelForSequenceClassification.from_pretrained(
    model_path,
    num_labels=1,
    quantization_config=bnb_config,
)

checkpoint_path = "../outputs/reasoning_reward_model/checkpoint-500"

model = PeftModel.from_pretrained(
    base_model,
    checkpoint_path,
)

model.eval()

chosen_scores = []
rejected_scores = []

with torch.no_grad():
    for example in tqdm(test_dataset, desc="测试奖励模型"):
        # test_dataset 中的 chosen / rejected 已经是应用过 chat template 的完整文本
        chosen_inputs = tokenizer(
            example["chosen"],
            return_tensors="pt",
            truncation=False,
            max_length=8192,
        ).to(model.device)

        rejected_inputs = tokenizer(
            example["rejected"],
            return_tensors="pt",
            truncation=False,
            max_length=8192,
        ).to(model.device)

        chosen_score = model(**chosen_inputs).logits.squeeze().float().item()
        rejected_score = model(**rejected_inputs).logits.squeeze().float().item()

        chosen_scores.append(chosen_score)
        rejected_scores.append(rejected_score)

# chosen 分数高于 rejected 分数，即该偏好对判断正确
correct_count = sum(
    chosen > rejected
    for chosen, rejected in zip(chosen_scores, rejected_scores)
)

preference_accuracy = correct_count / len(test_dataset)

print(f"测试集偏好对数量：{len(test_dataset)}")
print(f"判断正确数量：{correct_count}")
print(f"测试集偏好准确率：{preference_accuracy:.2%}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-0.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


测试奖励模型:   0%|          | 0/417 [00:00<?, ?it/s]

测试集偏好对数量：417
判断正确数量：264
测试集偏好准确率：63.31%


checkpoint 741 准确率：64.99%

checkpoint 700 准确率：64.51%

checkpoint 600 准确率：65.71%

checkpoint 500 准确率：63.31%

<div style="page-break-after: always;"></div>

# 2026/8/22

## PPO ( Proximal Policy Optimization) 近端策略优化

下面给出 **大语言模型 (LLM) 在 RLHF 场景下常用的 PPO 目标函数**——既包括标准 PPO 的“剪裁比值”式，也说明了 InstructGPT / TRL 等实现把 **KL 惩罚** 显式并入奖励的做法。记号一览：

| 记号                 | 含义                        |
| ------------------ | ------------------------- |
| $x$                | 提示／对话历史 (prompt)          |
| $y_{1:T}$          | 策略生成的序列                   |
| $s_t=(x,y_{<t})$   | 第 t 步的**状态**              |
| $\pi_\theta$       | 当前策略 (可训练 LLM)            |
| $\pi_{\text{old}}$ | 采样时冻结的旧策略                 |
| $\pi_{\text{ref}}$ | 参考模型（SFT 权重，梯度不更新）        |
| $R_\text{RM}(x,y)$ | 奖励模型给整条序列的打分              |
| $\beta$            | KL 系数（约 0.02 – 0.2）       |
| $\varepsilon$      | PPO clip 范围（典型 0.1 或 0.2） |

---

## A 奖励设计（sequence‑level + token‑level KL）

$$
\boxed{
r_t \;=\;
\begin{cases}
-\beta\,\bigl[\,
\log\pi_\theta(y_t\!\mid\!s_t)-\log\pi_{\text{ref}}(y_t\!\mid\!s_t)
\bigr] & \!1\le t<T\\[4pt]
R_\text{RM}(x,y)\;-\;
\beta\,\bigl[\,
\log\pi_\theta(y_T\!\mid\!s_T)-\log\pi_{\text{ref}}(y_T\!\mid\!s_T)
\bigr] & t=T
\end{cases}}
$$

* **终止步**加上来自人偏好或自动指标的标量奖励 $R_\text{RM}$。
* 每一步都累加 **KL 惩罚**，防止策略远离参考模型。

> *如果使用“KL 直接入损失”而不是放入 $r_t$*：见下节 2.2 的等价写法。

---

## B PPO 损失

### 2.1 剪裁比值式（OpenAI Spinning‑Up / TRL 默认）

1. **概率比值**

   $$
   \rho_t(\theta)=
   \frac{\pi_\theta(y_t\mid s_t)}{\pi_{\text{old}}(y_t\mid s_t)}
   $$

2. **优势估计**（GAE）

   $$
   A_t^{(\lambda)}=\sum_{k=0}^{\infty}
   (\gamma\lambda)^{k}\,
   \bigl[r_{t+k}
        +\gamma V_{\phi}(s_{t+k+1})
        -V_{\phi}(s_{t+k})\bigr]
   $$

3. **策略损失（剪裁）**

   $$
   \mathcal L_{\text{clip}}
   =-\;
   \mathbb E_t\Bigl[
     \min\bigl(
       \rho_t A_t,\;
       \operatorname{clip}(\rho_t,1-\varepsilon,1+\varepsilon)A_t
     \bigr)
   \Bigr]
   $$

4. **价值函数损失（可选 value‑clip）**

   $$
   \mathcal L_{V}
   =\tfrac12\,
    \mathbb E_t\Bigl[
      \bigl(
        V_{\phi}(s_t)-\hat G_t
      \bigr)^{\!2}
    \Bigr],\quad
   \hat G_t=\text{MonteCarlo return}
   $$

5. **熵正则**
   $\mathcal L_{\text{ent}}=-c_{\text{ent}}\;\mathbb E_t[\,H(\pi_\theta(\,\cdot\!\mid\!s_t))]$.

> **总损失** $\displaystyle
> \mathcal L =\mathcal L_{\text{clip}}
> +c_V\mathcal L_{V}
> +\mathcal L_{\text{ent}}
> $

这种写法把 **KL 惩罚当作奖励**，因此仍可用标准 PPO clip 公式直接优化。OpenAI InstructGPT 就是此体系([Cameron R. Wolfe][1])。


### PPO 训练流程与课件图解总结

PPO 可以理解为一个“模型生成回答—环境评价—策略更新”的闭环。在 RLHF 中，各部分的对应关系如下：

| 强化学习概念 | LLM/RLHF 中的对应内容 |
|---|---|
| 状态 $s_t$ | prompt 和已经生成的回答前缀 $y_{<t}$ |
| 动作 $a_t$ | 当前时刻生成的下一个 token |
| 策略 $\pi_\theta$ | 正在训练的语言模型 |
| 环境 | Reward Model 和人类偏好规则 |
| 奖励 $r_t$ | KL 惩罚，以及序列结束时的奖励模型分数 |

一次 PPO 外层迭代大致经过以下步骤：

1. **采样（rollout）**：使用当前策略 $\pi_\theta$ 对一批 prompt 生成回答，并记录状态、动作、奖励、旧策略的 token 概率以及 Value Model 的预测值。
2. **计算奖励**：每生成一个 token，就可以计算它和参考模型 $\pi_{ref}$ 的 KL 偏离；回答结束后，再把 Reward Model 对完整回答的分数加入最后一个 token。
3. **计算优势**：根据奖励和价值预测计算 return 与 advantage，判断某个动作相对于当前状态平均水平是好还是坏。
4. **策略更新**：利用 PPO 的概率比值和 clipping 机制更新策略，同时训练 Value Head。
5. **重复迭代**：丢弃本轮旧轨迹，用更新后的策略重新生成数据，持续提升回答质量。

---

### 1. KL 惩罚与稀疏奖励

课件中的 token 级奖励分配体现了两个设计：

- **逐 token 计算 KL**：对生成序列的每个 token，比较当前策略和参考模型的概率，偏离越大，惩罚越大；
- **序列级奖励稀疏分配**：中间 token 的 Reward Model 分数为 0，完整回答结束时才得到 $R_{RM}$。

因此最终奖励可以写成：

$$
R_{final}=R_{RM}-\beta\sum_{t=1}^{T}KL_t
$$

这样做的意义是兼顾两件事：一方面鼓励模型获得更高的整体回答质量，另一方面防止模型为了刷奖励而偏离原有语言能力和行为分布。奖励不应只看最后的分数，还要扣除过度偏离参考模型的代价。

---

### 2. 优势、Value Head 与 Value Loss

Value Head 估计当前状态未来能够获得的累计回报 $V(s_t)$。优势函数则衡量实际动作比模型预期好多少：

$$
A_t=Q(s_t,a_t)-V(s_t)
$$

实际实现通常使用 GAE（Generalized Advantage Estimation）估计优势，以降低方差并保持较好的偏差—方差平衡。Value Model 的训练目标是让预测值接近目标回报：

$$
V_{label}(s_t)=A_t^{GAE}+V(s_t)
$$

课件中介绍的三种目标构造方式：

| 方法 | 目标值 | 特点 |
|---|---|---|
| 蒙特卡洛 | $r_t+\gamma r_{t+1}+\gamma^2r_{t+2}+\cdots$ | 方差大 |
| 时序差分 | $r_t+\gamma V(s_{t+1})$ | 偏差较大 |
| 广义优势估计 | $A_t^{GAE}+V(s_t)$ | 综合平衡偏差与方差 |

为了防止 Value Head 一次更新幅度过大，PPO 还可以对新旧 value prediction 做 clipping，再取裁剪前后两种平方误差中的较大值计算 Value Loss。

---

### 3. PPO 的概率比值与 clipping

策略更新比较当前策略和采样时旧策略对同一个动作的概率：

$$
r_t(\theta)=\frac{\pi_\theta(a_t|s_t)}{\pi_{old}(a_t|s_t)}
$$

当这个比值离 1 太远时，说明新策略相对旧策略变化过大。PPO 将比值限制在 $[1-\varepsilon,1+\varepsilon]$ 的范围内：

$$
L^{CLIP}=-\mathbb{E}_t\left[\min\left(r_tA_t,\operatorname{clip}(r_t,1-\varepsilon,1+\varepsilon)A_t\right)\right]
$$

如果动作的优势为正，模型会提高它的概率，但提高幅度受到限制；如果优势为负，模型会降低它的概率，同样不会一步改变过多。clipping 是 PPO 稳定训练、避免策略崩溃的关键。

---

### 4. 外层迭代与内层训练

PPO 中有两个容易混淆的循环：

- **外层迭代（rollout iteration）**：用当前策略采样一批全新的 prompt—response 轨迹，计算奖励并冻结这些数据；
- **内层训练（PPO epochs）**：在同一批冻结数据上训练 $K$ 个 epoch，每轮重新打乱 minibatch，更新策略和 Value Head。

内层训练时，`logp_old` 和 `vpred_old` 保持不变，作为 clipping 的比较基准；`logp_new` 和 `vpred_new` 则在每个 minibatch 更新前重新计算。内层训练结束后，当前这批轨迹被丢弃，进入下一次外层采样。

> 核心记忆：**同一批 rollout 数据可以重复训练几轮，但不能无限重复使用；下一次外层迭代必须用更新后的策略重新采样。**

---

### 5. PPO 的计算代价与工程优化

RLHF 的训练速度通常明显慢于 SFT，主要瓶颈是每批数据都要重复进行 Value、Reward 和 KL 计算，尤其 Reward Model 需要对完整回答再次前向推理。常见问题包括：

- Reward Model 前向计算占用大量时间；
- 策略模型、参考模型、Value Head 和 Reward Model 可能同时占用显存；
- 每轮内层训练都需要重复计算新策略概率和价值预测。

可以考虑以下优化：

- 缓存已经计算过的 Reward 和 KL 结果，减少重复推理；
- 对多个样本进行批量处理，提高 GPU 利用率；
- 将不同模型分配到不同 GPU，或使用更小的 Reward Model；
- 适当减少 PPO 内层 epoch 数和生成长度；
- 使用梯度累积、量化和 LoRA 降低显存压力。

课件示例使用 `batch_size=2`、`mini_batch_size=1`、`ppo_epochs=3`，含义是每次外层采样 2 条数据，再将其拆成大小为 1 的 minibatch，在同一批冻结数据上更新 3 轮。

---

### 6. 与课件代码的对应关系

```python
ppo_config = PPOConfig(
    kl_penalty="full",  # 对当前策略与参考策略施加 KL 约束
    ppo_epochs=3,          # 同一批 rollout 数据的内层训练轮数
    batch_size=2,          # 每次外层采样/训练的样本数
    mini_batch_size=1,     # 每次参数更新实际使用的 minibatch 大小
)
```

生成回答后，将 query 和 response 拼接，交给奖励模型得到分数：

```python
input_ids = torch.concat([query, response], dim=0)
input_ids = input_ids.unsqueeze(0)
score = ppo_trainer.model.compute_reward_score(
    input_ids=input_ids
)[0, -1, 0]
```

随后把这些分数传给 `step()`，PPO 才能根据奖励、KL 和优势更新策略。整体流程可以记为：

$$
prompt \rightarrow \text{policy generate} \rightarrow \text{Reward Model scoring} \rightarrow \text{KL penalty} \rightarrow \text{advantage} \rightarrow \text{PPO update}
$$